In [1]:
import os
import sys
import json
import glob
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "/home/iec/MinhHieu/Non-Invasive/rPPG"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.iBVPNet import iBVPNet
from neural_methods.model.FactorizePhys.FactorizePhys import FactorizePhys

Matplotlib is building the font cache; this may take a moment.


In [2]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data/Normal")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/Normal/groupF")
OUTPUT_DIR          = os.path.join(REPO_ROOT, "results/Normal/groupF")

# ----- video / signal params -----
VIDEO_FPS   = 30       # camera frame rate
PPG_FS      = 60       # PPG sensor sampling rate (Hz)

# ----- preprocessing params -----
CHUNK_LENGTH = 160     # frames per clip
IMG_H, IMG_W = 72, 72  # face crop resolution
LABEL_TYPE   = "Standardized"  # z-score label (no cumsum in post-processing)

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Device: cuda:0
PREPROCESSED_PATH: /home/iec/MinhHieu/Non-Invasive/rPPG/preprocessed_data/Normal/groupF
OUTPUT_DIR: /home/iec/MinhHieu/Non-Invasive/rPPG/results/Normal/groupF


In [ ]:
# Model toggle

MODELS = [
    # ("PURE_iBVPNet",              "iBVPNet",       "final_model_release/PURE_iBVPNet.pth"),
    # ("PURE_FactorizePhys",        "FactorizePhys", "final_model_release/PURE_FactorizePhys_FSAM_Res.pth"),
    # ("iBVP_FactorizePhys",      "FactorizePhys", "final_model_release/iBVP_FactorizePhys_FSAM_Res.pth"),
    # ("SCAMPS_FactorizePhys",    "FactorizePhys", "final_model_release/SCAMPS_FactorizePhys_FSAM_Res.pth"),
    # ("UBFC-rPPG_FactorizePhys", "FactorizePhys", "final_model_release/UBFC-rPPG_FactorizePhys_FSAM_Res.pth"),
    ("MyData_FactorizePhys",     "FactorizePhys", "final_model_release/GroupF_FactorizePhys.pth.pth"),
]

print(f"Will run {len(MODELS)} model(s):")
for name, arch, path in MODELS:
    print(f"  {name}  ({arch})  ->  {path}")

Will run 5 model(s):
  PURE_iBVPNet  (iBVPNet)  ->  final_model_release/PURE_iBVPNet.pth
  PURE_FactorizePhys  (FactorizePhys)  ->  final_model_release/PURE_FactorizePhys_FSAM_Res.pth
  iBVP_FactorizePhys  (FactorizePhys)  ->  final_model_release/iBVP_FactorizePhys_FSAM_Res.pth
  SCAMPS_FactorizePhys  (FactorizePhys)  ->  final_model_release/SCAMPS_FactorizePhys_FSAM_Res.pth
  UBFC-rPPG_FactorizePhys  (FactorizePhys)  ->  final_model_release/UBFC-rPPG_FactorizePhys_FSAM_Res.pth


In [ ]:
# Read video frames

def read_video_frames(video_path):
    """Read all frames from a video file (MP4, MKV, AVI, etc.).

    Returns:
        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)

In [5]:
def read_ppg_synced(session_path, num_frames):
    """
    Reads the PPG signal from 'ppg.csv' and resamples it to match the exact 
    timestamps of the video frames from 'frame_timestamps.csv'.
    """
    import pandas as pd
    import numpy as np
    import os
    
    # 1. Read the video frame timestamps
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    
    # Validation
    if len(frame_df) != num_frames:
        print(f"Warning: Video has {num_frames} frames, but frame_timestamps.csv has {len(frame_df)} rows. Using min count.")
        min_len = min(len(frame_df), num_frames)
        frame_t = frame_df["timestamp"].values[:min_len]
    else:
        frame_t = frame_df["timestamp"].values

    # 2. Read the raw PPG data
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    
    # Clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    
    # 3. Resample (Interpolate)
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    
    return ppg_resampled.astype(np.float32)

In [6]:
# Normalization functions
# DATA_TYPE is Raw: no normalization on input frames (just face crop + resize)
# LABEL_TYPE is Standardized: z-score normalization on labels

def standardized_label(label):
    """Standardized label: global z-score."""
    label = label.astype(np.float64)
    m = np.mean(label)
    s = np.std(label)
    if s > 0:
        label = (label - m) / s
    else:
        label = np.zeros_like(label)
    return label.astype(np.float32)

In [7]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [ ]:
# Discover subjects

all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_path = subj_dir

    video_pattern = os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv")
    video_files = glob.glob(video_pattern)
    
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
    })
    print(f"  {subj_id}  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

Found 10 subject folders

  S_001  video=S_001.mkv
  S_002  video=S_002.mkv
  S_003  video=S_003.mkv
  S_004  video=S_004.mkv
  S_005  video=S_005.mkv
  S_006  video=S_006.mkv
  S_007  video=S_007.mkv
  S_008  video=S_008.mkv
  S_009  video=S_009.mkv
  S_010  video=S_010.mkv

Total subjects: 10


In [9]:
# Data preprocessing
# DATA_TYPE = Raw: face crop + resize only, NO normalization on input frames
# LABEL_TYPE = Standardized: z-score on labels
# Save as .npy with shape (CHUNK_LENGTH, H, W, 3)

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    ppg_signal = read_ppg_synced(session_path, T)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # Face crop + resize (Raw data type: no normalization on frames)
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)  # (T, 72, 72, 3) float32

    # Standardized label (z-score)
    label = standardized_label(ppg_signal)  # (T,)

    # Chunk into clips of CHUNK_LENGTH
    clip_num = T // CHUNK_LENGTH
    frame_clips = np.array([frames_cropped[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([label[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH] for i in range(clip_num)])

    # Save per-subject subfolder
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        np.save(input_path, frame_clips[chunk_idx])  # (CHUNK_LENGTH, H, W, 3)
        np.save(label_path, label_clips[chunk_idx])   # (CHUNK_LENGTH,)
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*_input*.npy")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

Cleared and recreated: /home/iec/MinhHieu/Non-Invasive/rPPG/preprocessed_data/Normal/groupF

=== Processing S001 ===
  Video: 2703 frames @ 30 fps
  PPG green: min=1, max=127
  16 clips -> /home/iec/MinhHieu/Non-Invasive/rPPG/preprocessed_data/Normal/groupF/S001

=== Processing S002 ===
  Video: 2700 frames @ 30 fps
  PPG green: min=1, max=91
  16 clips -> /home/iec/MinhHieu/Non-Invasive/rPPG/preprocessed_data/Normal/groupF/S002

=== Processing S003 ===
  Video: 2704 frames @ 30 fps
  PPG green: min=1, max=127
  16 clips -> /home/iec/MinhHieu/Non-Invasive/rPPG/preprocessed_data/Normal/groupF/S003

=== Processing S004 ===
  Video: 2702 frames @ 30 fps
  PPG green: min=1, max=127
  16 clips -> /home/iec/MinhHieu/Non-Invasive/rPPG/preprocessed_data/Normal/groupF/S004

=== Processing S005 ===
  Video: 2702 frames @ 30 fps
  PPG green: min=1, max=104
  16 clips -> /home/iec/MinhHieu/Non-Invasive/rPPG/preprocessed_data/Normal/groupF/S005

=== Processing S006 ===
  Video: 2702 frames @ 30 fps

In [13]:
# PyTorch Dataset + DataLoader
# DATA_FORMAT: NCDHW -> transpose from (T, H, W, C) to (C, T, H, W)

class GroupFDataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))  # (T,)

        # NDHWC -> NCDHW: transpose (T, H, W, C) -> (C, T, H, W)
        data = np.transpose(data, (3, 0, 1, 2))  # (3, T, H, W)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = GroupFDataset(all_input_files)
print(f"Dataset: {len(dataset)} clips")

loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=4)
print(f"DataLoader ready: {len(loader)} batches")

Dataset: 160 clips
DataLoader ready: 40 batches


In [14]:
# Post-processing functions
# LABEL_TYPE = Standardized -> diff_flag=False (NO cumsum)

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending (Tarvainen et al.)."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones  = np.ones(T_len)
    D_mat = (np.diag(ones[:-2], -2)
             - 2 * np.diag(ones[:-1], -1)
             + np.diag(ones))
    D_mat = D_mat[2:, :]
    inv   = np.linalg.inv(H_mat + lambda_val ** 2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=1):
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low / fs * 2, high / fs * 2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz(sig, fs, low, high):
    """Return dominant frequency (Hz) in [low, high] Hz via FFT."""
    N = 1
    while N < len(sig):
        N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any():
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise (dB)."""
    N = 1
    while N < len(pred_ppg):
        N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)

    f1  = hr_label_bpm / 60.0
    f2  = 2 * f1
    dev = 6.0 / 60.0  # +-6 bpm tolerance

    sig_mask   = (((freqs >= f1 - dev) & (freqs <= f1 + dev))
                  | ((freqs >= f2 - dev) & (freqs <= f2 + dev)))
    noise_mask = ((freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask)

    sig_power   = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0:
        return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    """Concatenate chunks in sorted key order into a 1-D array."""
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])


def process_bvp(pred_chunks, label_chunks, fs=30, diff_flag=False):
    """Post-process BVP predictions and labels.

    diff_flag=False for Standardized labels (no cumsum needed).
    """
    pred  = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)

    if diff_flag:
        pred  = detrend(np.cumsum(pred),  100)
        label = detrend(np.cumsum(label), 100)
    else:
        pred  = detrend(pred,  100)
        label = detrend(label, 100)

    pred_processed  = bandpass_filter(pred,  fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)

    hr_pred  = fft_peak_hz(pred_processed,  fs, 0.6, 3.3) * 60.0
    hr_label = fft_peak_hz(label_processed, fs, 0.6, 3.3) * 60.0
    snr_db   = calculate_snr(pred_processed, hr_label, fs)

    return hr_pred, hr_label, snr_db, pred_processed

In [17]:
# Inference loop with model-specific forward, export to results_groupF/

def build_model(arch, model_path):
    """Instantiate model by architecture name and load weights."""
    full_path = os.path.join(REPO_ROOT, model_path)
    
    # Thêm weights_only=True để ẩn cảnh báo bảo mật
    state_dict = torch.load(full_path, map_location=DEVICE, weights_only=False)

    # strip 'module.' prefix if present (DataParallel artifact)
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    if arch == "iBVPNet":
        model = iBVPNet(frames=CHUNK_LENGTH, in_channels=3)
        model.load_state_dict(state_dict)
    elif arch == "FactorizePhys":
        MD_CONFIG = {
            "FRAME_NUM": CHUNK_LENGTH,
            "MD_FSAM": True,
            "MD_TYPE": "NMF",
            "MD_TRANSFORM": "T_KAB",
            "MD_R": 1,
            "MD_S": 1,
            "MD_STEPS": 3,
            "MD_INFERENCE": True,
            "MD_RESIDUAL": True,
        }
        model = FactorizePhys(
            frames=CHUNK_LENGTH,
            md_config=MD_CONFIG,
            in_channels=3,
            dropout=0.1,
            device=torch.device(DEVICE),
        )
        model.load_state_dict(state_dict, strict=False)
    else:
        raise ValueError(f"Unknown architecture: {arch}")

    model = model.to(DEVICE)
    model.eval()
    return model


def run_inference(model, arch, loader):
    """Run inference and return per-subject prediction/label dicts."""
    preds_dict  = {}  # subj_key -> {chunk_id: np.ndarray (CHUNK_LENGTH,)}
    labels_dict = {}

    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            data_t, labels_t, batch_subjects, batch_chunk_ids = batch

            # data_t shape: (N, C, T, H, W) = NCDHW
            data_in = data_t.float().to(DEVICE)

            # Both models use torch.diff internally and need T+1 frames.
            # Pad by repeating last frame in temporal dim.
            last_frame = data_in[:, :, -1:, :, :].clone()
            data_padded = torch.cat([data_in, last_frame], dim=2)  # (N, 3, T+1, H, W)

            if arch == "iBVPNet":
                pred_ppg = model(data_padded)  # (N, T)
            elif arch == "FactorizePhys":
                out = model(data_padded)
                pred_ppg = out[0]  # (N, T)
            else:
                raise ValueError(f"Unknown architecture: {arch}")

            pred_np  = pred_ppg.cpu().numpy()                    # (N, T)
            label_np = labels_t.numpy()                          # (N, T)

            N = data_t.shape[0]
            for i in range(N):
                subj = batch_subjects[i]
                cid  = int(batch_chunk_ids[i])

                if subj not in preds_dict:
                    preds_dict[subj]  = {}
                    labels_dict[subj] = {}

                preds_dict[subj][cid]  = pred_np[i]   # (T,)
                labels_dict[subj][cid] = label_np[i]  # (T,)

    return preds_dict, labels_dict


# Run all models
FS = VIDEO_FPS

for model_name, arch, model_path in MODELS:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}  ({arch})")
    print(f"Weights: {model_path}")
    print(f"{'='*60}")

    model = build_model(arch, model_path)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {num_params:,}")

    preds_dict, labels_dict = run_inference(model, arch, loader)
    print(f"Subjects: {sorted(preds_dict.keys())}")

    # Per-subject results
    per_subject_results = []
    hr_preds_all  = []
    hr_labels_all = []
    snr_all       = []

    print(f"\n{'Subject':<10} {'HR_pred':>10} {'HR_label':>10} {'HR_err':>8} {'SNR':>7}")
    print("-" * 55)

    for subj_key in sorted(preds_dict.keys()):
        hr_pred, hr_label, _, pred_processed = process_bvp(
            preds_dict[subj_key], labels_dict[subj_key], fs=FS, diff_flag=False
        )

        snr_db = calculate_snr(pred_processed, hr_label, FS)
        hr_err = hr_pred - hr_label

        # map subj_key ("S000") back to original ID ("S_000")
        subj_id = subj_key[0] + "_" + subj_key[1:]

        per_subject_results.append({
            "name":                subj_id,
            "predicted_heartrate": hr_pred,
            "label_heartrate":     hr_label,
            "heartrate_error":     hr_err,
            "snr_db":              snr_db,
        })

        hr_preds_all.append(hr_pred)
        hr_labels_all.append(hr_label)
        snr_all.append(snr_db)

        print(f"{subj_id:<10} {hr_pred:>10.3f} {hr_label:>10.3f} {hr_err:>8.3f} {snr_db:>7.2f}")

    hr_preds_all  = np.array(hr_preds_all)
    hr_labels_all = np.array(hr_labels_all)
    snr_all       = np.array(snr_all)

    # Aggregate metrics
    n = len(hr_preds_all)
    assert n > 0, "No subjects to evaluate."

    err   = hr_preds_all - hr_labels_all
    abs_e = np.abs(err)
    sq_e  = err ** 2
    rel_e = abs_e / (np.abs(hr_labels_all) + 1e-9)

    mae       = float(np.mean(abs_e))
    mae_se    = float(np.std(abs_e) / np.sqrt(n))
    rmse      = float(np.sqrt(np.mean(sq_e)))
    rmse_se   = float(np.sqrt(np.std(sq_e) / np.sqrt(n)))
    mape      = float(np.mean(rel_e) * 100.0)
    mape_se   = float(np.std(rel_e) / np.sqrt(n) * 100.0)

    if n >= 2:
        pearson_r  = float(np.corrcoef(hr_preds_all, hr_labels_all)[0, 1])
        pearson_se = float(np.sqrt(max(0.0, (1 - pearson_r ** 2) / (n - 2))))
    else:
        pearson_r, pearson_se = float("nan"), float("nan")

    mean_snr    = float(np.mean(snr_all))
    mean_snr_se = float(np.std(snr_all) / np.sqrt(n))

    print(f"\nAggregate metrics for {model_name}:")
    print(f"  MAE     : {mae:.4f} +/- {mae_se:.4f} bpm")
    print(f"  RMSE    : {rmse:.4f} +/- {rmse_se:.4f} bpm")
    print(f"  MAPE    : {mape:.4f} +/- {mape_se:.4f} %")
    print(f"  Pearson : {pearson_r:.4f} +/- {pearson_se:.4f}")
    print(f"  SNR     : {mean_snr:.4f} +/- {mean_snr_se:.4f} dB")

    # Export results
    metrics_dict = {
        "model":      model_name,
        "architecture": arch,
        "n_subjects": n,
        "evaluation_method": "FFT BVP-derived HR",
        "label_type": LABEL_TYPE,
        "bvp_bandpass_hz":   [0.6, 3.3],
        "aggregate_metrics": {
            "MAE":     {"value": mae,       "se": mae_se,      "unit": "bpm"},
            "RMSE":    {"value": rmse,      "se": rmse_se,     "unit": "bpm"},
            "MAPE":    {"value": mape,      "se": mape_se,     "unit": "%"},
            "Pearson": {"value": pearson_r, "se": pearson_se, "unit": ""},
            "SNR":     {"value": mean_snr,  "se": mean_snr_se, "unit": "dB"},
        },
        "per_subject": [
            {
                "name":                r["name"],
                "predicted_heartrate": r["predicted_heartrate"],
                "label_heartrate":     r["label_heartrate"],
                "heartrate_error":     r["heartrate_error"],
                "snr_db":              r["snr_db"],
            }
            for r in per_subject_results
        ],
    }

    model_output_dir = os.path.join(OUTPUT_DIR, model_name)
    os.makedirs(model_output_dir, exist_ok=True)

    json_path = os.path.join(model_output_dir, "metrics.json")
    with open(json_path, "w") as fh:
        json.dump(metrics_dict, fh, indent=2)
    print(f"\n  Metrics saved to: {json_path}")

    csv_rows = []
    for r in per_subject_results:
        csv_rows.append({
            "name":                r["name"],
            "predicted_heartrate": r["predicted_heartrate"],
            "label_heartrate":     r["label_heartrate"],
            "heartrate_error":     r["heartrate_error"],
        })

    results_df = pd.DataFrame(csv_rows, columns=[
        "name", "predicted_heartrate", "label_heartrate", "heartrate_error"
    ])

    csv_path = os.path.join(model_output_dir, "ppg_results.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"  CSV saved to: {csv_path}")
    print()
    print(results_df.to_string(index=False))

    # Clean up model from GPU
    del model
    torch.cuda.empty_cache()

print(f"\n\nAll models complete. Results in: {OUTPUT_DIR}")


Model: PURE_iBVPNet  (iBVPNet)
Weights: final_model_release/PURE_iBVPNet.pth
Total parameters: 1,426,865


Inference: 100%|██████████| 40/40 [00:29<00:00,  1.35it/s]


Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          81.738     83.057   -1.318   -1.32
S_002          85.693     85.693    0.000    2.45
S_003          81.738     81.738    0.000    1.71
S_004          56.689     93.164  -36.475   -4.92
S_005          86.572     86.133    0.439    4.50
S_006          62.842     62.842    0.000    4.77
S_007          90.967     90.967    0.000    1.44
S_008          63.721     63.721    0.000    3.24
S_009          71.631     71.631    0.000    4.34
S_010          78.662     79.102   -0.439    4.48

Aggregate metrics for PURE_iBVPNet:
  MAE     : 3.8672 +/- 3.4394 bpm
  RMSE    : 11.5435 +/- 11.2334 bpm
  MAPE    : 4.1804 +/- 3.6893 %
  Pearson : 0.4673 +/- 0.3126
  SNR     : 2.0694 +/- 0.9298 dB

  Metrics saved to: /home/iec/MinhHieu/Non-Invasive/rPPG/results/Normal/groupF/PURE_iBVPNet/metrics

Inference: 100%|██████████| 40/40 [00:05<00:00,  7.37it/s]


Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          88.770     83.057    5.713   -0.87
S_002          85.693     85.693    0.000    5.10
S_003          81.738     81.738    0.000    4.13
S_004          93.164     93.164    0.000    1.52
S_005          86.572     86.133    0.439    7.44
S_006          62.842     62.842    0.000    3.29
S_007          90.967     90.967    0.000    5.99
S_008          63.721     63.721    0.000    2.55
S_009          71.631     71.631    0.000    4.20
S_010          78.662     79.102   -0.439    6.36

Aggregate metrics for PURE_FactorizePhys:
  MAE     : 0.6592 +/- 0.5355 bpm
  RMSE    : 1.8172 +/- 1.7585 bpm
  MAPE    : 0.7944 +/- 0.6447 %
  Pearson : 0.9863 +/- 0.0583
  SNR     : 3.9719 +/- 0.7421 dB

  Metrics saved to: /home/iec/MinhHieu/Non-Invasive/rPPG/results/Normal/groupF/PURE_FactorizePh

...

Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          86.133     83.057    3.076    1.13
S_002          85.693     85.693    0.000    6.12
S_003          81.738     81.738    0.000    5.38
S_004          93.164     93.164    0.000    4.45
S_005          86.572     86.133    0.439    7.90
S_006          62.842     62.842    0.000    6.11
S_007          90.967     90.967    0.000    6.49
S_008          63.721     63.721    0.000    3.26
S_009          71.631     71.631    0.000    5.10
S_010          79.102     79.102    0.000    7.65

Aggregate metrics for iBVP_FactorizePhys:
  MAE     : 0.3516 +/- 0.2902 bpm
  RMSE    : 0.9826 +/- 0.9465 bpm
  MAPE    : 0.4214 +/- 0.3493 %
  Pearson : 0.9960 +/- 0.0315
  SNR     : 5.3598 +/- 0.6110 dB

  Metrics saved to: /home/iec/MinhHieu/Non-Invasive/rPPG/results/Normal/groupF/iBVP_FactorizePh

Inference: 100%|██████████| 40/40 [00:01<00:00, 29.06it/s]


Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          81.738     83.057   -1.318   -2.51
S_002          85.693     85.693    0.000    3.11
S_003          81.738     81.738    0.000    1.53
S_004          93.164     93.164    0.000   -1.16
S_005          86.133     86.133    0.000    5.08
S_006          63.721     62.842    0.879   -1.31
S_007          90.967     90.967    0.000    4.12
S_008          63.721     63.721    0.000    1.41
S_009          71.631     71.631    0.000   -0.11
S_010          78.662     79.102   -0.439    3.23

Aggregate metrics for SCAMPS_FactorizePhys:
  MAE     : 0.2637 +/- 0.1417 bpm
  RMSE    : 0.5200 +/- 0.4135 bpm
  MAPE    : 0.3541 +/- 0.1879 %
  Pearson : 0.9988 +/- 0.0171
  SNR     : 1.3402 +/- 0.7660 dB

  Metrics saved to: /home/iec/MinhHieu/Non-Invasive/rPPG/results/Normal/groupF/SCAMPS_Factori

Inference: 100%|██████████| 40/40 [00:01<00:00, 30.90it/s]


Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          83.057     83.057    0.000   -0.39
S_002          85.693     85.693    0.000    4.63
S_003          81.738     81.738    0.000    4.10
S_004          93.164     93.164    0.000    2.95
S_005          86.572     86.133    0.439    6.96
S_006          62.842     62.842    0.000    2.54
S_007          90.967     90.967    0.000    6.10
S_008          63.721     63.721    0.000    2.00
S_009          71.631     71.631    0.000    3.22
S_010          79.102     79.102    0.000    5.94

Aggregate metrics for UBFC-rPPG_FactorizePhys:
  MAE     : 0.0439 +/- 0.0417 bpm
  RMSE    : 0.1390 +/- 0.1354 bpm
  MAPE    : 0.0510 +/- 0.0484 %
  Pearson : 0.9999 +/- 0.0045
  SNR     : 3.8050 +/- 0.6636 dB

  Metrics saved to: /home/iec/MinhHieu/Non-Invasive/rPPG/results/Normal/groupF/UBFC-rPPG_F

In [ ]:
# convert metric.json to csv

import os
import glob
import json
import pandas as pd

# 1. Khai báo đường dẫn gốc chứa các thư mục model dựa trên ảnh của bạn
ROOT_DIR = OUTPUT_DIR

# 2. Tìm tất cả các file metrics.json nằm trong các thư mục con
json_files = glob.glob(os.path.join(ROOT_DIR, "*", "metrics.json"))

data_rows = []

# 3. Lặp qua từng file JSON để lấy dữ liệu
for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
        model_name = data.get("model", "Unknown")
        n_subjects = data.get("n_subjects", 0)
        metrics = data.get("aggregate_metrics", {})
        
        # Lấy giá trị MAE gốc để làm tiêu chí sắp xếp (Rank)
        mae_raw = metrics.get("MAE", {}).get("value", float('inf'))
        
        # Hàm định dạng chữ theo chuẩn "value +/- se" (làm tròn 2 chữ số)
        def format_metric(m):
            if not m: return ""
            return f"{m.get('value', 0):.2f} +/- {m.get('se', 0):.2f}"

        # Đẩy dữ liệu vào 1 hàng (row)
        row = {
            "model": model_name,
            "# subjects": n_subjects,
            "MAE_raw": mae_raw, # Cột tạm để sort
            "MAE (bpm)": format_metric(metrics.get("MAE")),
            "RMSE (bpm)": format_metric(metrics.get("RMSE")),
            "MAPE (%)": format_metric(metrics.get("MAPE")),
            "Pearson": format_metric(metrics.get("Pearson")),
            "SNR (dB)": format_metric(metrics.get("SNR")),
        }
        data_rows.append(row)

# 4. Chuyển thành DataFrame (bảng)
df = pd.DataFrame(data_rows)

if not df.empty:
    # Sắp xếp bảng theo giá trị MAE thô (từ thấp nhất -> cao nhất)
    df = df.sort_values(by="MAE_raw", ascending=True).reset_index(drop=True)
    
    # Thêm cột 'rank' vào vị trí đầu tiên (bắt đầu từ 1)
    df.insert(0, "rank", df.index + 1)
    
    # Xóa cột 'MAE_raw' vì không cần hiển thị ra CSV
    df = df.drop(columns=["MAE_raw"])
    
    # 5. Xuất ra file CSV
    out_csv_path = os.path.join(ROOT_DIR, "Model_Performance_Metrics.csv")
    df.to_csv(out_csv_path, index=False)
    
    print(f"✅ Đã gom thành công {len(json_files)} file JSON!")
    print(f"✅ File tổng hợp được lưu tại:\n{out_csv_path}\n")
    print("Preview dữ liệu:")
    print(df.head().to_string(index=False))
else:
    print("❌ Không tìm thấy file metrics.json nào trong thư mục!")